In [ ]:
import os
from pathlib import Path
import logging
import shutil
from datetime import datetime
import time

import pandas as pd
from pyspark.sql import functions as F

import teehr
from teehr.evaluation.evaluation import RemoteReadOnlyEvaluation
from teehr.evaluation.spark_session_utils import create_spark_session
from teehr.fetching.usgs.usgs import usgs_to_parquet
from teehr.fetching.utils import create_periods_based_on_chunksize, get_period_start_end_times

logger = logging.getLogger()

- NWM v3.0: 2023-09-19 - present
- Earliest reference time in our secondary table: 2025-11-10 22:00:00

In [ ]:
BASE_DIR = Path("/data/data_processing/backfill-usgs")
BASE_DIR.mkdir(parents=True, exist_ok=True)

LOCATIONS_IDS_FILEPATH = Path(BASE_DIR, "usgs_conus_ids.csv")

#### One-Time: Get USGS location IDs using RemoteReadOnly and save to disk

In [ ]:
%%time
spark = create_spark_session()  # For read-only   --> Change this to read-write when backfilling
ev = teehr.RemoteReadOnlyEvaluation(spark=spark)

In [ ]:
%%time
locations_df = ev.location_attributes_view(
        attr_list=["is_active", "has_inst_discharge"]
).filter(
    filters=[
        {
            "column": "location_id",
            "operator": "like",
            "value": "usgs-%"
        },
        "is_active = 'True'",
        "has_inst_discharge = 'True'"
    ]
).to_pandas()

In [ ]:
locations_df.to_csv(LOCATIONS_IDS_FILEPATH, index=False)

In [ ]:
spark.stop()

#### Now read in the USGS location IDs from disk

Local Read Write

In [ ]:
%%time
dir_path =  Path(BASE_DIR, "usgs-teehr-warehouse")
spark = create_spark_session() 
ev = teehr.LocalReadWriteEvaluation(
    spark=spark,
    dir_path=dir_path,
    create_dir=True
)

In [ ]:
locations_df = pd.read_csv(LOCATIONS_IDS_FILEPATH)

sites = locations_df["location_id"].str.upper().to_list()
len(sites)

In [ ]:
# Break usgs_sites into chunks
CHUNK_SIZE = 100
usgs_site_chunks = [
    sites[i:i + CHUNK_SIZE]
    for i in range(0, len(sites), CHUNK_SIZE)
]

usgs_variable_name = "streamflow_hourly_inst"
output_parquet_dir = Path(
    ev.fetch.usgs_cache_dir,
    "usgs_observations",
    usgs_variable_name
)

service="iv"
chunk_by="month"
filter_to_hourly=True
filter_no_data=True
convert_to_si=True
overwrite_output=False

output_parquet_dir

In [ ]:
start_date = datetime(2026, 4, 1, 0)  # 2023-09-19 is the start of nwm v3.0
end_date = datetime(2026, 4, 15, 0)    

In [ ]:
import os
os.environ["API_USGS_PAT"] = "API KEY HERE"  # setting as env variable in terminal did not seem to work for me

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
MAX_WORKERS_CAP = 6

In [ ]:
%%time
"""
Process: Sends a request for CHUNK_SIZE num IDs over a time span of chunk_by for each sequential iteration.

"""
def _fetch_chunk(
    i,
    site_chunk,
    dts,
    output_parquet_dir,
    service,
    filter_to_hourly,
    filter_no_data,
    convert_to_si,
    overwrite_output
):  
    # # TEMP!
    # if i <= 74:
    #     return
        
    usgs_to_parquet(
        sites=site_chunk,
        start_date=dts["start_dt"],
        end_date=dts["end_dt"],
        output_parquet_dir=Path(output_parquet_dir, f"{dts["start_dt"].strftime("%Y%m%d%H")}_{dts["end_dt"].strftime("%Y%m%d%H")}", f"site_chunk_{i}"),
        service=service,
        chunk_by=None,
        filter_to_hourly=filter_to_hourly,
        filter_no_data=filter_no_data,
        convert_to_si=convert_to_si,
        overwrite_output=overwrite_output
    )
        
periods = create_periods_based_on_chunksize(
    start_date=start_date,
    end_date=end_date,
    chunk_by=chunk_by
)

for period_num, period in enumerate(periods):
        
    if period is not None:
        dts = get_period_start_end_times(
            period=period,
            start_date=start_date,
            end_date=end_date
        )
    else:
        dts = {"start_dt": start_date, "end_dt": end_date}

    t0 = time.time()
    logger.info(f"START: Fetching data between dates {dts["start_dt"]} and {dts["end_dt"]}.")
    # Concurrently: (seems to use tokens more?) WAY FASTER
    with ThreadPoolExecutor(max_workers=min(len(usgs_site_chunks), MAX_WORKERS_CAP)) as executor:
        futures = {
            executor.submit(
                _fetch_chunk, i, site_chunk, dts, output_parquet_dir, service,
                filter_to_hourly, filter_no_data, convert_to_si, overwrite_output
            ): i
            for i, site_chunk in enumerate(usgs_site_chunks)
        }
        for future in as_completed(futures):
            i = futures[future]
            try:
                future.result()
                logger.info(f"Fetched chunk: {i}") 
            except Exception as e:
                logger.error(f"Chunk {i} failed: {e}")  
    # # Sequentially:
    # for i, site_chunk in enumerate(usgs_site_chunks):
    #     _fetch_chunk(
    #         i=i,
    #         site_chunk=site_chunk,
    #         dts=dts,
    #         output_parquet_dir=output_parquet_dir,
    #         service=service,
    #         filter_to_hourly=filter_to_hourly,
    #         filter_no_data=filter_no_data,
    #         convert_to_si=convert_to_si,
    #         overwrite_output=overwrite_output             
    #     )
    logger.info(f"END: Fetched data between dates {dts["start_dt"]} and {dts["end_dt"]} in: {(time.time() - t0) / 60:.2f}min.")

    break